# Seawaste Dataset Exploration

This notebook helps you explore and validate your seawaste detection dataset.

In [ ]:
import sys
sys.path.append('..')

import os
import yaml
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd
from collections import Counter

from utils.data_utils import verify_dataset_structure, MarineDataPreprocessor

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 8)

## 1. Load Dataset Configuration

In [ ]:
# Load dataset configuration
config_path = '../configs/seawaste_dataset.yaml'

with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

print("Dataset Configuration:")
print(f"Number of classes: {config['nc']}")
print(f"\nClass names:")
for idx, name in config['names'].items():
    print(f"  {idx}: {name}")

## 2. Verify Dataset Structure

In [ ]:
# Verify dataset
stats = verify_dataset_structure(config_path)

print("\nDataset Statistics:")
print(f"Total images: {stats['total_images']}")
print(f"Total labels: {stats['total_labels']}")
print(f"\nSplit breakdown:")
print(f"  Train: {stats['train_images']} images, {stats['train_labels']} labels")
print(f"  Val:   {stats['val_images']} images, {stats['val_labels']} labels")
print(f"  Test:  {stats['test_images']} images, {stats['test_labels']} labels")

## 3. Visualize Sample Images

In [ ]:
def visualize_sample_with_boxes(image_path, label_path, class_names):
    """Visualize image with bounding boxes"""
    # Read image
    img = cv2.imread(str(image_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    
    # Read labels
    boxes = []
    if label_path.exists():
        with open(label_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                class_id = int(parts[0])
                x_center, y_center, width, height = map(float, parts[1:])
                boxes.append((class_id, x_center, y_center, width, height))
    
    # Draw boxes
    img_with_boxes = img.copy()
    for class_id, x_center, y_center, box_w, box_h in boxes:
        # Convert to pixel coordinates
        x1 = int((x_center - box_w/2) * w)
        y1 = int((y_center - box_h/2) * h)
        x2 = int((x_center + box_w/2) * w)
        y2 = int((y_center + box_h/2) * h)
        
        # Draw rectangle
        cv2.rectangle(img_with_boxes, (x1, y1), (x2, y2), (0, 255, 0), 2)
        
        # Put label
        label = class_names.get(class_id, f'Class {class_id}')
        cv2.putText(img_with_boxes, label, (x1, y1-5), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
    
    return img_with_boxes, len(boxes)

# Get sample images from train set
dataset_path = Path(config['path'])
train_images_path = dataset_path / config['train']

if train_images_path.exists():
    image_files = list(train_images_path.glob('*.jpg')) + list(train_images_path.glob('*.png'))
    
    if len(image_files) > 0:
        # Show first 6 samples
        n_samples = min(6, len(image_files))
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        axes = axes.flatten()
        
        for idx in range(n_samples):
            img_path = image_files[idx]
            label_path = dataset_path / 'labels' / 'train' / f"{img_path.stem}.txt"
            
            img_with_boxes, n_boxes = visualize_sample_with_boxes(
                img_path, label_path, config['names']
            )
            
            axes[idx].imshow(img_with_boxes)
            axes[idx].set_title(f'{img_path.name}\n{n_boxes} objects')
            axes[idx].axis('off')
        
        plt.tight_layout()
        plt.show()
    else:
        print("No images found in training set")
else:
    print(f"Training images path not found: {train_images_path}")
    print("Please add your dataset to the data/datasets/seawaste/ directory")

## 4. Analyze Class Distribution

In [ ]:
def analyze_class_distribution(labels_dir, class_names):
    """Analyze class distribution in dataset"""
    class_counts = Counter()
    
    label_files = list(Path(labels_dir).glob('*.txt'))
    
    for label_file in label_files:
        with open(label_file, 'r') as f:
            for line in f:
                class_id = int(line.strip().split()[0])
                class_counts[class_id] += 1
    
    return class_counts

# Analyze train set
train_labels_dir = dataset_path / 'labels' / 'train'

if train_labels_dir.exists():
    class_counts = analyze_class_distribution(train_labels_dir, config['names'])
    
    # Create dataframe
    df_classes = pd.DataFrame([
        {'Class ID': cls_id, 
         'Class Name': config['names'].get(cls_id, f'Unknown'),
         'Count': count}
        for cls_id, count in sorted(class_counts.items())
    ])
    
    print("\nClass Distribution (Training Set):")
    print(df_classes.to_string(index=False))
    
    # Visualize
    plt.figure(figsize=(14, 6))
    plt.bar(df_classes['Class Name'], df_classes['Count'])
    plt.xlabel('Class')
    plt.ylabel('Number of Instances')
    plt.title('Class Distribution in Training Set')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print("No labels found in training set")

## 5. Test Marine Preprocessing

In [ ]:
# Test marine image preprocessing
preprocessor = MarineDataPreprocessor()

if len(image_files) > 0:
    # Load a sample image
    sample_img_path = image_files[0]
    img = cv2.imread(str(sample_img_path))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Apply preprocessing
    img_enhanced = preprocessor.enhance_underwater_image(img)
    img_enhanced_rgb = cv2.cvtColor(img_enhanced, cv2.COLOR_BGR2RGB)
    
    # Visualize comparison
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    
    axes[0].imshow(img_rgb)
    axes[0].set_title('Original Image')
    axes[0].axis('off')
    
    axes[1].imshow(img_enhanced_rgb)
    axes[1].set_title('Enhanced (Marine Preprocessing)')
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()
else:
    print("No images available for preprocessing demo")

## 6. Image Size Analysis

In [ ]:
# Analyze image sizes
if len(image_files) > 0:
    sizes = []
    for img_path in image_files[:100]:  # Sample first 100 images
        img = cv2.imread(str(img_path))
        if img is not None:
            h, w = img.shape[:2]
            sizes.append((w, h))
    
    if sizes:
        df_sizes = pd.DataFrame(sizes, columns=['Width', 'Height'])
        
        print("\nImage Size Statistics:")
        print(df_sizes.describe())
        
        # Plot size distribution
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        axes[0].hist(df_sizes['Width'], bins=20, alpha=0.7, label='Width')
        axes[0].hist(df_sizes['Height'], bins=20, alpha=0.7, label='Height')
        axes[0].set_xlabel('Pixels')
        axes[0].set_ylabel('Frequency')
        axes[0].set_title('Image Dimension Distribution')
        axes[0].legend()
        
        axes[1].scatter(df_sizes['Width'], df_sizes['Height'], alpha=0.5)
        axes[1].set_xlabel('Width (pixels)')
        axes[1].set_ylabel('Height (pixels)')
        axes[1].set_title('Image Aspect Ratios')
        axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
else:
    print("No images available for size analysis")

## Summary

This notebook helped you:
1. ✓ Load and verify dataset configuration
2. ✓ Visualize sample images with annotations
3. ✓ Analyze class distribution
4. ✓ Test marine-specific preprocessing
5. ✓ Analyze image sizes and aspect ratios

Next steps:
- Review class balance (consider augmentation or resampling if highly imbalanced)
- Ensure image sizes are compatible with training (640x640 is standard)
- Verify annotation quality
- Proceed to training!